In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense, BatchNormalization, Bidirectional
from tensorflow.keras.metrics import Recall

import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)
tf.random.set_seed(seed)

2026-04-07 11:29:57.820037: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775561398.018127      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775561398.084684      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775561398.609302      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775561398.609339      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775561398.609341      17 computation_placer.cc:177] computation placer alr

In [3]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val-2"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train.npy'))
X_val   = np.load(os.path.join(INPUT_PATH, 'X_val.npy'))
X_test  = np.load(os.path.join(INPUT_PATH, 'X_test.npy'))

y_train_sepsis6 = np.load(os.path.join(INPUT_PATH, 'y_train_sepsis6.npy'))
y_val_sepsis6   = np.load(os.path.join(INPUT_PATH, 'y_val_sepsis6.npy'))
y_test_sepsis6  = np.load(os.path.join(INPUT_PATH, 'y_test_sepsis6.npy'))

y_train_vital = np.load(os.path.join(INPUT_PATH, 'y_train_vital.npy'))
y_val_vital   = np.load(os.path.join(INPUT_PATH, 'y_val_vital.npy'))
y_test_vital  = np.load(os.path.join(INPUT_PATH, 'y_test_vital.npy'))

y_train_vital_mask = np.load(os.path.join(INPUT_PATH, 'y_train_vital_mask.npy'))
y_val_vital_mask   = np.load(os.path.join(INPUT_PATH, 'y_val_vital_mask.npy'))
y_test_vital_mask  = np.load(os.path.join(INPUT_PATH, 'y_test_vital_mask.npy'))

In [4]:
X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train_sepsis6 = np.asarray(y_train_sepsis6).astype(np.float32)
y_val_sepsis6   = np.asarray(y_val_sepsis6).astype(np.float32)
y_test_sepsis6  = np.asarray(y_test_sepsis6).astype(np.float32)

y_train_vital = np.asarray(y_train_vital).astype(np.float32)
y_val_vital   = np.asarray(y_val_vital).astype(np.float32)
y_test_vital  = np.asarray(y_test_vital).astype(np.float32)

y_train_vital_mask = np.asarray(y_train_vital_mask).astype(np.float32)
y_val_vital_mask   = np.asarray(y_val_vital_mask).astype(np.float32)
y_test_vital_mask  = np.asarray(y_test_vital_mask).astype(np.float32)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train_sepsis6:", y_train_sepsis6.shape, y_train_sepsis6.dtype)
print("y_train_vital:", y_train_vital.shape, y_train_vital.dtype)
print("y_train_vital_mask:", y_train_vital_mask.shape, y_train_vital_mask.dtype)

assert X_train.ndim == 3
assert X_val.ndim == 3
assert X_test.ndim == 3

assert len(X_train) == len(y_train_sepsis6) == len(y_train_vital) == len(y_train_vital_mask)
assert len(X_val) == len(y_val_sepsis6) == len(y_val_vital) == len(y_val_vital_mask)
assert len(X_test) == len(y_test_sepsis6) == len(y_test_vital) == len(y_test_vital_mask)

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("n_horizon =", y_train_sepsis6.shape[1])
print("n_vital =", y_train_vital.shape[1])

# dùng cho head vital
y_train_vital_pack = np.concatenate([y_train_vital, y_train_vital_mask], axis=1).astype(np.float32)
y_val_vital_pack   = np.concatenate([y_val_vital, y_val_vital_mask], axis=1).astype(np.float32)
y_test_vital_pack  = np.concatenate([y_test_vital, y_test_vital_mask], axis=1).astype(np.float32)

print("y_train_vital_pack:", y_train_vital_pack.shape)

X_train: (119140, 10, 133) float32
y_train_sepsis6: (119140, 6) float32
y_train_vital: (119140, 8) float32
y_train_vital_mask: (119140, 8) float32
timesteps = 10
n_features = 133
n_horizon = 6
n_vital = 8
y_train_vital_pack: (119140, 16)


In [5]:
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.models import Model

N_FEATURES = X_train.shape[2]
N_HORIZON = y_train_sepsis6.shape[1]   # 6

flat_y = y_train_sepsis6.reshape(-1)
n_pos = np.sum(flat_y == 1)
n_neg = np.sum(flat_y == 0)
POS_WEIGHT = float(n_neg / max(n_pos, 1))
print("POS_WEIGHT =", POS_WEIGHT)

def weighted_bce_6(y_true, y_pred):
    eps = K.epsilon()
    y_pred = tf.clip_by_value(y_pred, eps, 1. - eps)
    loss = -(POS_WEIGHT * y_true * tf.math.log(y_pred) +
             (1. - y_true) * tf.math.log(1. - y_pred))
    return tf.reduce_mean(loss)

def create_multihorizon_model(n_lstm_units, dropout_rate):
    inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))

    x = Bidirectional(
        LSTM(n_lstm_units, return_sequences=True)
    )(inputs)
    x = Dropout(dropout_rate)(x)
    x = BatchNormalization()(x)

    x = Bidirectional(
        LSTM(max(n_lstm_units // 2, 8), return_sequences=False)
    )(x)
    x = Dropout(dropout_rate)(x)
    x = BatchNormalization()(x)

    shared = Dense(16, activation='relu')(x)
    shared = Dropout(0.2)(shared)

    sepsis_out = Dense(N_HORIZON, activation='sigmoid', name='sepsis')(shared)

    model = Model(inputs=inputs, outputs=sepsis_out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=weighted_bce_6,
        metrics=[
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )
    return model

print("✅ Đã khởi tạo create_multihorizon_model()")

POS_WEIGHT = 13.38888888888889
✅ Đã khởi tạo create_multihorizon_model()


In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow.keras.backend as K

K.clear_session()

BEST_UNITS = 16
BEST_DROPOUT = 0.3
BEST_BATCH = 512

final_model = create_multihorizon_model(
    n_lstm_units=BEST_UNITS,
    dropout_rate=BEST_DROPOUT
)

early_stopping = EarlyStopping(
    monitor='val_auprc',
    mode='max',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_auprc',
    mode='max',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_lstm_multihorizon_model.keras',
    monitor='val_auprc',
    mode='max',
    save_best_only=True,
    verbose=1
)

history = final_model.fit(
    X_train,
    y_train_sepsis6,
    validation_data=(X_val, y_val_sepsis6),
    epochs=50,
    batch_size=BEST_BATCH,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    shuffle=True,
    verbose=1
)

2026-04-07 11:30:24.972458: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - auprc: 0.1207 - auroc: 0.6346 - loss: 1.3084 - precision: 0.0977 - recall: 0.6449
Epoch 1: val_auprc improved from -inf to 0.26096, saving model to best_lstm_multihorizon_model.keras
233/233 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - auprc: 0.1209 - auroc: 0.6350 - loss: 1.3077 - precision: 0.0978 - recall: 0.6452 - val_auprc: 0.2610 - val_auroc: 0.8078 - val_loss: 1.0166 - val_precision: 0.1868 - val_recall: 0.7180 - learning_rate: 0.0010
Epoch 2/50
232/233 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - auprc: 0.2701 - auroc: 0.8354 - loss: 0.9555 - precision: 0.1885 - recall: 0.7972
Epoch 2: val_auprc improved from 0.26096 to 0.27303, saving model to best_lstm_multihorizon_model.keras
233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - auprc: 0.2701 - auroc: 0.8355 - loss: 0.9551 - precision: 0.1886 - recall: 0.7972 - val_auprc: 0.2730 - val_auroc: 0.8260 - val_loss: 1.0195 - val_precision: 0.1996 - val_recall: 0.7275 - learning_rate: 0.0010
Epoch 3/50
2

In [7]:
val_pred_sepsis = final_model.predict(X_val, batch_size=BEST_BATCH, verbose=0)
test_pred_sepsis = final_model.predict(X_test, batch_size=BEST_BATCH, verbose=0)

val_prob_t6 = val_pred_sepsis[:, 5]
test_prob_t6 = test_pred_sepsis[:, 5]

y_val_t6 = y_val_sepsis6[:, 5].astype(int)
y_test_t6 = y_test_sepsis6[:, 5].astype(int)

In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

rows = []
for th in np.arange(0.05, 0.96, 0.01):
    val_pred = (val_prob_t6 >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val_t6, val_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    youden_j = sensitivity + specificity - 1

    rows.append({
        "threshold": round(th, 2),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "youden_j": youden_j
    })

df_th = pd.DataFrame(rows)
best = df_th.loc[df_th["youden_j"].idxmax()]
print(best)

threshold      0.360000
sensitivity    0.806648
specificity    0.721274
youden_j       0.527922
Name: 31, dtype: float64


In [9]:
candidate = df_th[df_th["sensitivity"] >= 0.80].copy()
best = candidate.sort_values("specificity", ascending=False).iloc[0]
print(best)

threshold      0.370000
sensitivity    0.801378
specificity    0.725487
youden_j       0.526865
Name: 32, dtype: float64


In [10]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score
)

best_threshold = float(best["threshold"])

# xác suất -> nhãn
test_pred = (test_prob_t6 >= best_threshold).astype(int)

# confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test_t6, test_pred).ravel()

# threshold-based metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
precision = precision_score(y_test_t6, test_pred, zero_division=0)
recall = recall_score(y_test_t6, test_pred, zero_division=0)
f1 = f1_score(y_test_t6, test_pred, zero_division=0)
acc = accuracy_score(y_test_t6, test_pred)
youden_j = sensitivity + specificity - 1

# probability-based metrics
auroc = roc_auc_score(y_test_t6, test_prob_t6)
auprc = average_precision_score(y_test_t6, test_prob_t6)

print(f"=== TEST RESULT WITH THRESHOLD = {best_threshold:.2f} ===")
print(f"AUROC:       {auroc:.4f}")
print(f"AUPRC:       {auprc:.4f}")
print(f"Accuracy:    {acc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Youden's J:  {youden_j:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_t6, test_pred))

print("\nClassification Report:")
print(classification_report(y_test_t6, test_pred, digits=4))

=== TEST RESULT WITH THRESHOLD = 0.37 ===
AUROC:       0.8541
AUPRC:       0.3275
Accuracy:    0.7396
Sensitivity: 0.8186
Specificity: 0.7332
Precision:   0.1991
Recall:      0.8186
F1-score:    0.3204
Youden's J:  0.5518

Confusion Matrix:
[[25280  9201]
 [  507  2288]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9803    0.7332    0.8389     34481
           1     0.1991    0.8186    0.3204      2795

    accuracy                         0.7396     37276
   macro avg     0.5897    0.7759    0.5796     37276
weighted avg     0.9218    0.7396    0.8000     37276

